In [1]:
!pip install -q transformers accelerate pypdf
!pip install -U bitsandbytes

from transformers import AutoTokenizer, AutoModelForCausalLM
from pypdf import PdfReader
import torch, re, json
from pathlib import Path

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 12.5 MB/s eta 0:00:00


In [2]:
from google.colab import files
from pathlib import Path
import re, json

# Option A: upload PDF from your machine
print("Upload your PDF…")
up = files.upload()  # pick your file in the dialog
pdf_name = next(iter(up.keys()))
PDF_PATH = Path(pdf_name)

Upload your PDF…


Saving rag_base.pdf to rag_base.pdf


In [3]:
from pypdf import PdfReader

def pdf_to_text(path: Path) -> str:
    reader = PdfReader(str(path))
    return "\n\n".join((p.extract_text() or "") for p in reader.pages)

def clean_text(t: str) -> str:
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)                 # join hyphen line-breaks
    t = re.sub(r"[ \t]*\n(?!\s*\n)", " ", t)               # collapse single newlines
    t = re.sub(r"\s+\n", "\n", t)
    return t.strip()

raw = pdf_to_text(PDF_PATH)
text = clean_text(raw)

# Simple "chapter" split: split on common headings; fallback to big chunks if no headings
parts = re.split(r"(?i)(?:\n\s*(?:Kapitel|Chapter)\s+\d+\b|^\s*\d+(?:\.\d+)*\s+[^\n]+$)", text, flags=re.M)
parts = [p.strip() for p in parts if len(p.split()) > 80]  # keep only substantive parts
print(f"Segments detected: {len(parts)}")


Segments detected: 68


In [4]:
# --- Load Qwen chat model in 4-bit and a helper to do chat-style generation ---
import torch, json, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb)

def chat_generate(system_prompt: str, user_prompt: str,
                  max_new_tokens=640, temperature=0.4, do_sample=True):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
# ===== Robust Q&A generation with content filtering, debug, and fallback =====
from pathlib import Path
import re, json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

OUT_QA = Path("domain_qa.jsonl")

# 0) (Re)load Qwen chat model in 4-bit (if not already loaded in this runtime)
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=bnb)

def chat_generate(system_prompt: str, user_prompt: str,
                  max_new_tokens=720, temperature=0.4, do_sample=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt}
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=0.9,
        repetition_penalty=1.05,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

# 1) Filter out front-matter / promo / letters
BAD_SIGNALS = (
    "Liebe Leserin, lieber Leser", "Bundesminister", "kostenlos herausgegeben",
    "Wahlwerbern", "Europa-, Bundestags-", "Wahl", "Vorwort", "Grußwort"
)
GOOD_KEYWORDS = (
    "Versicherung", "Krankenversicherung", "Beitrag", "Pflicht", "Bemessungsgrenze",
    "Mitglied", "GKV", "Krankenkasse", "Selbstverwaltung", "Leistung", "Richtlinie",
    "Versicherungsfreiheit", "Pflichtversichert", "freiwillig", "PKV"
)

def is_content_segment(s: str) -> bool:
    s_low = s.lower()
    if any(k.lower() in s_low for k in BAD_SIGNALS):
        # if it strongly looks like front-matter, drop unless it also has many good terms
        if sum(1 for k in GOOD_KEYWORDS if k.lower() in s_low) < 2:
            return False
    # keep only segments with substance
    return len(s.split()) >= 80 and sum(1 for k in GOOD_KEYWORDS if k.lower() in s_low) >= 1

segments = [p.strip() for p in parts if is_content_segment(p.strip())]
if not segments:
    # fallback: naive fixed-size chunks from the full text
    segments = []
    step = 2500
    for i in range(0, len(text), step):
        seg = text[i:i+step]
        if is_content_segment(seg):
            segments.append(seg)

print(f"Using {len(segments)} content segments (filtered from {len(parts)}).")

# 2) Prompts + few-shot
FEWSHOT_JSON = (
    '[{"question":"Was bedeutet Versicherungspflicht?",'
    '"answer":"Sie ist die gesetzliche Verpflichtung, krankenversichert zu sein."},'
    '{"question":"Wer ist pflichtversichert?",'
    '"answer":"Arbeitnehmer, Studierende und Rentner, sofern gesetzliche Kriterien erfüllt sind."}]'
)

def make_prompts(section: str, n: int = 3):
    section = section[:2800]  # allow a bit more context
    system = (
        "Du erstellst aus einem deutschen Sachtext sinnvolle Prüfungsfragen mit Antworten. "
        "Nutze ausschließlich Informationen aus dem Abschnitt. Antworte nur als JSON-Liste."
    )
    user = (
        f"Erstelle genau {n} Frage-Antwort-Paare (Q&A) AUF DEUTSCH aus dem folgenden Abschnitt.\n"
        "Regeln:\n"
        " - Fragen müssen prüfbar sein (Definitionen, Pflichten, Verfahren, Kennzahlen, Zuständigkeiten).\n"
        " - Antworten kurz (1–3 Sätze) und nur aus dem Abschnitt (keine externen Quellen).\n"
        " - GIB AUSSCHLIESSLICH eine JSON-LISTE im Format aus dem Beispiel zurück.\n"
        f"Beispiel:\n{FEWSHOT_JSON}\n\n"
        f"Abschnitt:\n{section}\n"
    )
    return system, user

# 3) JSON extractor + quality filter
def extract_json_list(s: str):
    if "[" not in s or "]" not in s:
        return None
    blob = s[s.find("["):s.rfind("]")+1]
    try:
        data = json.loads(blob)
        fixed=[]
        for x in data:
            q = x.get("question") or x.get("frage") or x.get("Frage") or x.get("Q")
            a = x.get("answer")   or x.get("antwort") or x.get("Antwort") or x.get("A")
            if q and a:
                fixed.append({"question": str(q).strip(), "answer": str(a).strip()})
        return fixed
    except Exception:
        return None

PRONOUN_BAD = {"dies", "dieses", "diese", "das", "es", "sie", "er", "unser", "unsere",
               "unserer", "ihr", "ihre", "grundsätzlich", "in deutschland"}
Q_START_OK = ("Was ", "Welche ", "Wie ", "Wer ", "Worin ", "Woraus ", "Weshalb ", "Wozu ")

def is_meaningful_qa(qa: dict) -> bool:
    q = (qa.get("question") or "").strip()
    a = (qa.get("answer") or "").strip()
    if not q.endswith("?"): return False
    if not (12 <= len(q) <= 220): return False
    if not (20 <= len(a) <= 800): return False
    if not q.startswith(Q_START_OK): return False
    m = re.match(r"Was\s+ist\s+(.+?)\?\s*$", q, flags=re.I)
    if m:
        x = re.sub(r"[^\wÄÖÜäöüß\- ]", "", m.group(1)).strip().lower()
        if (x in PRONOUN_BAD) or (len(x) < 4 and " " not in x):
            return False
    return True

# 4) Fallback: extract QAs from segments that already contain question-like headings
def fallback_qas_from_text(section: str, max_qas=3):
    qas=[]
    # take lines/sentences that end with '?' and grab the next sentence(s) as answer
    sents = [s.strip() for s in re.split(r"(?<=[\.\?\!…])\s+", section) if s.strip()]
    for i, s in enumerate(sents):
        if s.endswith("?") and any(s.startswith(st) for st in Q_START_OK):
            # answer = next 1–2 sentences that are not another question
            ans_parts=[]
            j=i+1
            while j < len(sents) and not sents[j].endswith("?") and len(ans_parts) < 2:
                ans_parts.append(sents[j]); j+=1
            if ans_parts:
                qa={"question": s, "answer": " ".join(ans_parts)}
                if is_meaningful_qa(qa):
                    qas.append(qa)
        if len(qas) >= max_qas: break
    return qas

# 5) Debug helper: peek raw model output for a segment index
def debug_segment(idx: int, n=3, temp=0.4, do_sample=False):
    sec = segments[idx]
    sys, usr = make_prompts(sec, n=n)
    out = chat_generate(sys, usr, temperature=temp, do_sample=do_sample)
    print("=== RAW MODEL OUTPUT ===\n", out[:1600], "\n========================")
    return out

# 6) Generate
all_qas, seen = [], set()
for i, sec in enumerate(segments[:30], 1):
    # try deterministic first
    sys, usr = make_prompts(sec, n=3)
    out = chat_generate(sys, usr, temperature=0.4, do_sample=False)
    print(out)
    items = extract_json_list(out) or []

    # if empty, retry with sampling
    if not items:
        out2 = chat_generate(sys, usr, temperature=0.7, do_sample=True)
        items = extract_json_list(out2) or []

    # final fallback: rule-based extraction from in-text questions
    if not items:
        items = fallback_qas_from_text(sec, max_qas=3)

    # keep only meaningful + unique
    kept_before = len(all_qas)
    for qa in items:
        if is_meaningful_qa(qa):
            key = (qa["question"].lower(), qa["answer"].lower())
            if key not in seen:
                seen.add(key)
                all_qas.append(qa)

    print(f"Segment {i}: model_raw={len(extract_json_list(out) or [])} "
          f"model_retry={len(extract_json_list(out2) or []) if 'out2' in locals() else 0} "
          f"fallback={len(fallback_qas_from_text(sec))} "
          f"(kept {len(all_qas)} total; +{len(all_qas)-kept_before} new)")

with OUT_QA.open("w", encoding="utf-8") as f:
    for qa in all_qas:
        f.write(json.dumps(qa, ensure_ascii=False) + "\n")

print(f"Wrote {len(all_qas)} Q&As → {OUT_QA}")

Using 63 content segments (filtered from 68).
Segment 1: model_raw=0 model_retry=0 fallback=0 (kept 0 total; +0 new)
Segment 2: model_raw=0 model_retry=0 fallback=0 (kept 0 total; +0 new)
